## Silver Layer – Data Standardisation & Validation

The Silver layer transforms raw Bronze data into a **trusted, analysis-ready dataset**.
At this stage, the focus is on:
- Standardising formats and data types
- Recalculating unreliable source metrics
- Flagging data quality issues
- Applying core business rules (SLA logic)

No aggregations or analytical modelling are performed in Silver.


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from datetime import datetime
from pyspark.sql import Row


StatementMeta(, fd90a551-f7cd-47fb-91b2-c0dce1b08396, 4, Finished, Available, Finished)

In [3]:
print("=" * 60)
print("SILVER LAYER TRANSFORMATION - STARTED")
print("=" * 60)

# ============================================
# CELL 1: Watermark Initialization
# ============================================

# Watermark schema
watermark_schema = StructType([
    StructField("table_name", StringType(), False), 
    StructField("watermark_value", TimestampType(), False)
])

# Watermark column mapping
WATERMARK_COLUMNS = {
    "cleaned_bronze_network_events": "Outage_Start",
}

def initialize_watermark_table():
    """Initialize or get existing watermark table"""
    try:
        df_watermark = spark.table("lh_Silver_Telecom.dbo.watermarktable")
        print("Watermark table exists")
        return df_watermark
    except:
        print("Creating watermark table...")
        initial_data = [("cleaned_bronze_network_events", datetime(2010, 1, 1, 0, 0, 0))]
        df_watermark = spark.createDataFrame(initial_data, watermark_schema)
        df_watermark.write.format("delta").mode("overwrite") \
            .saveAsTable("lh_Silver_Telecom.dbo.watermarktable")
        print("Watermark table created")
        return df_watermark

def get_current_watermark(table_name):
    """Get current watermark value"""
    try:
        watermark_df = spark.table("lh_Silver_Telecom.dbo.watermarktable")
        watermark_row = watermark_df.filter(col("table_name") == table_name).collect()
        if watermark_row:
            return watermark_row[0]["watermark_value"]
        else:
            return datetime(2010, 1, 1, 0, 0, 0)
    except:
        return datetime(2010, 1, 1, 0, 0, 0)

def update_watermark(table_name, new_watermark_value):
    """Update watermark after successful processing"""
    try:
        new_watermark_data = [(table_name, new_watermark_value)]
        new_watermark_df = spark.createDataFrame(new_watermark_data, watermark_schema)
        new_watermark_df.createOrReplaceTempView("new_watermark_temp")
        
        spark.sql(f"""
            MERGE INTO lh_Silver_Telecom.dbo.watermarktable AS target
            USING new_watermark_temp AS source
            ON target.table_name = source.table_name
            WHEN MATCHED THEN 
                UPDATE SET target.watermark_value = source.watermark_value
            WHEN NOT MATCHED THEN 
                INSERT (table_name, watermark_value) 
                VALUES (source.table_name, source.watermark_value)
        """)
        print(f"Watermark updated to {new_watermark_value}")
        return True
    except Exception as e:
        print(f"Watermark update failed: {str(e)}")
        return False

# Initialize watermark
watermark_table = initialize_watermark_table()


StatementMeta(, fd90a551-f7cd-47fb-91b2-c0dce1b08396, 5, Finished, Available, Finished)

SILVER LAYER TRANSFORMATION - STARTED
Watermark table exists


## Read from Bronze Table

This step reads raw network event data from the Bronze Delta table.
The Silver layer always consumes **tables**, not raw files, to ensure a clean contract between layers.


In [4]:
# ============================================
# Read Incremental Data from Bronze
# ============================================

table_name = "cleaned_bronze_network_events"
current_watermark = get_current_watermark(table_name)
watermark_column = WATERMARK_COLUMNS[table_name]

print(f"\nCurrent watermark: {current_watermark}")

# Read only new data since last watermark
bronze_df = spark.sql(f"""
    SELECT * 
    FROM lh_Silver_Telecom.dbo.cleaned_bronze_network_events
    WHERE {watermark_column} > '{current_watermark}'
""")

row_count = bronze_df.count()
print(f"New records to process: {row_count}")

if row_count == 0:
    print("No new data to process. Silver layer up to date.")
else:
    print("Proceeding with Silver transformation...")


StatementMeta(, fd90a551-f7cd-47fb-91b2-c0dce1b08396, 6, Finished, Available, Finished)


Current watermark: 2024-04-28 23:47:00
New records to process: 0
No new data to process. Silver layer up to date.


SystemExit: No new data

In [19]:
# ============================================
# Date/Time Enrichment
# ============================================

print("\ Adding date/time dimensions...")

silver_df = (
    bronze_df
    # Date columns for dimension joins
    .withColumn("outage_start_date", to_date(col("Outage_Start")))
    .withColumn("outage_end_date", to_date(col("Outage_End")))
    
    # Time-based attributes
    .withColumn("outage_start_hour", hour(col("Outage_Start")))
    .withColumn("outage_end_hour", hour(col("Outage_End")))
    .withColumn("outage_day_of_week", dayofweek(col("Outage_Start")))
    
    # Business context flags
    .withColumn("is_business_hours", 
        (hour(col("Outage_Start")) >= 8) & (hour(col("Outage_Start")) <= 17))
    .withColumn("is_weekend",
        dayofweek(col("Outage_Start")).isin([1, 7]))
    
    # Partitioning columns for Gold layer
    .withColumn("outage_year", year(col("Outage_Start")))
    .withColumn("outage_month", month(col("Outage_Start")))
    .withColumn("outage_quarter", quarter(col("Outage_Start")))
)

print("Date/time enrichment complete")

StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 21, Finished, Available, Finished)

\ Adding date/time dimensions...
Date/time enrichment complete


In [20]:
# ============================================
# Duration Validation & Correction
# ============================================

print("\ Validating and correcting durations...")

silver_df = (
    silver_df
    # Validate duration
    .withColumn("is_negative_duration", col("Calculated_Duration") < 0)
    .withColumn("is_missing_end_time", col("Outage_End").isNull())
    .withColumn("is_excessive_duration", col("Calculated_Duration") > 10080)  # > 7 days
    
    # Correct duration
    .withColumn("duration_minutes_corrected",
        when(col("Calculated_Duration") < 0, abs(col("Calculated_Duration")))
        .when(col("Calculated_Duration") > 10080, None)  # Treat > 7 days as error
        .otherwise(col("Calculated_Duration"))
    )
    
    # Categorize duration for analysis
    .withColumn("duration_category",
        when(col("duration_minutes_corrected").isNull(), "Invalid")
        .when(col("duration_minutes_corrected") < 15, "Minor (< 15 min)")
        .when(col("duration_minutes_corrected") < 60, "Moderate (15-60 min)")
        .when(col("duration_minutes_corrected") < 240, "Major (1-4 hrs)")
        .otherwise("Critical (> 4 hrs)")
    )
)

print("Duration validation complete")


StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 22, Finished, Available, Finished)

\ Validating and correcting durations...
Duration validation complete


## Standardise Text Fields

Text-based columns are normalised to ensure consistency across records.
This includes:
- Trimming leading and trailing whitespace
- Standardising casing where appropriate

These changes do not alter business meaning and are safe for Silver.


In [21]:
#Normalize string columns

silver_df = (
    silver_df
    .withColumn("Site_ID", trim(col("Site_ID")))
    .withColumn("Province", trim(col("Province")))
    .withColumn("Province_Code", upper(trim(col("Province_Code"))))
    .withColumn("Vendor", trim(col("Vendor")))
    .withColumn("Technology", upper(trim(col("Technology"))))
    .withColumn("Cause", trim(col("Cause")))
)




StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 23, Finished, Available, Finished)

## Data Quality Validation Flags

Instead of removing records, Silver flags potential data quality issues:
- Negative outage durations
- Missing outage end timestamps
- Invalid availability percentages

This approach preserves transparency and supports downstream investigation.


## SLA Compliance Logic

A row-level SLA indicator is applied based on a 99.999% availability target.
This enables:
- Identification of potential SLA breaches
- Aggregated SLA reporting in the Gold layer

Final SLA evaluation is deferred to later stages.


In [22]:
# ============================================
# Availability Validation
# ============================================

print("\ Validating availability metrics...")

silver_df = (
    silver_df
    .withColumn("is_invalid_availability",
        (col("Availability_Percent") < 0) | (col("Availability_Percent") > 100)
    )
    
    # Correct invalid availability (set to NULL for investigation)
    .withColumn("availability_percent_corrected",
        when((col("Availability_Percent") < 0) | (col("Availability_Percent") > 100), None)
        .otherwise(col("Availability_Percent"))
    )
    
    # Flag potential SLA breaches (per-event, not true SLA)
    .withColumn("is_sla_breach_indicator",
        col("availability_percent_corrected") < 99.9
    )
)

print("Availability validation complete")


StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 24, Finished, Available, Finished)

\ Validating availability metrics...
Availability validation complete


In [23]:
# ============================================
# Add Audit Columns
# ============================================

print("\ Adding audit columns...")

silver_df = (
    silver_df
    .withColumn("silver_processed_timestamp", current_timestamp())
    .withColumn("silver_load_id", lit(f"load_{datetime.now().strftime('%Y%m%d_%H%M%S')}"))
)

print("Audit columns added")

StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 25, Finished, Available, Finished)

\ Adding audit columns...
Audit columns added


In [24]:
# ============================================
# Data Quality Summary
# ============================================

print("\n" + "=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

dq_checks = silver_df.select(
    count("*").alias("total_records"),
    sum(when(col("is_negative_duration"), 1).otherwise(0)).alias("negative_duration_count"),
    sum(when(col("is_missing_end_time"), 1).otherwise(0)).alias("missing_end_time_count"),
    sum(when(col("is_invalid_availability"), 1).otherwise(0)).alias("invalid_availability_count"),
    sum(when(col("is_excessive_duration"), 1).otherwise(0)).alias("excessive_duration_count")
).collect()[0]


print(f"Total records: {dq_checks['total_records']}")
print(f"Negative durations: {dq_checks['negative_duration_count']}")
print(f"Missing end times: {dq_checks['missing_end_time_count']}")
print(f"Invalid availability: {dq_checks['invalid_availability_count']}")
print(f"Excessive durations (> 7 days): {dq_checks['excessive_duration_count']}")


StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 26, Finished, Available, Finished)


DATA QUALITY SUMMARY
Total records: 10627
Negative durations: 0
Missing end times: 0
Invalid availability: 476
Excessive durations (> 7 days): 0


In [25]:
# ============================================
# Write to Silver Table
# ============================================

print("\n" + "=" * 60)
print("WRITING TO SILVER TABLE")
print("=" * 60)

# Write to Silver (append mode for incremental)
(
    silver_df
    .write
    .format("delta")
    .mode("append")  
    .option("mergeSchema", "true")  # Allow schema evolution
    .saveAsTable("lh_Silver_Telecom.dbo.silver_network_events")
)

print(f"{row_count} records written to silver_network_events")

StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 27, Finished, Available, Finished)


WRITING TO SILVER TABLE
10627 records written to silver_network_events


In [26]:
# ============================================
# Data Quality Metrics
# ============================================
wm_year = current_watermark.year
wm_month = current_watermark.month

dq_summary = spark.createDataFrame([Row(
    check_timestamp=datetime.now(),
    outage_year=wm_year,
    outage_month=wm_month,
    total_records=dq_checks['total_records'] or 0,
    negative_duration_count=dq_checks['negative_duration_count'] or 0,
    missing_end_time_count=dq_checks['missing_end_time_count'] or 0,
    invalid_availability_count=dq_checks['invalid_availability_count'] or 0,
    excessive_duration_count=dq_checks['excessive_duration_count'] or 0
)])

dq_summary.write.format("delta").mode("append") \
    .saveAsTable("lh_Silver_Telecom.dbo.silver_data_quality_metrics")

print("Data quality metrics recorded")

StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 28, Finished, Available, Finished)

Data quality metrics recorded


In [27]:
# ============================================
# Update Watermark
# ============================================

max_watermark = silver_df.agg(max(watermark_column)).collect()[0][0]

if max_watermark:
    update_watermark(table_name, max_watermark)
    print(f"Processing complete - watermark updated to {max_watermark}")
else:
    print("⚠ Warning: No valid watermark found in processed data")


StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 29, Finished, Available, Finished)

Watermark updated to 2024-04-28 23:47:00
Processing complete - watermark updated to 2024-04-28 23:47:00


In [28]:
# ============================================
# Validation & Sample Output
# ============================================

print("\n" + "=" * 60)
print("SILVER TABLE VALIDATION")
print("=" * 60)

silver_validation = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

print(f"Total records in Silver: {silver_validation.count()}")

print("\nSample records:")
silver_validation.select(
    "Site_ID",
    "Vendor",
    "outage_start_date",
    "duration_minutes_corrected",
    "duration_category",
    "is_sla_breach_indicator"
).show(10, truncate=False)

print("\nDuration category distribution:")
silver_validation.groupBy("duration_category").count().orderBy(desc("count")).show()


print("=" * 60)
print("SILVER LAYER TRANSFORMATION - COMPLETED")
print("=" * 60)

StatementMeta(, 66537295-e85c-4fdc-83e0-04563e76186c, 30, Finished, Available, Finished)


SILVER TABLE VALIDATION
Total records in Silver: 10627

Sample records:
+------------+------+-----------------+--------------------------+--------------------+-----------------------+
|Site_ID     |Vendor|outage_start_date|duration_minutes_corrected|duration_category   |is_sla_breach_indicator|
+------------+------+-----------------+--------------------------+--------------------+-----------------------+
|SA-SITE-0084|VEND_C|2024-04-03       |76                        |Major (1-4 hrs)     |true                   |
|SA-SITE-0431|VEND_C|2024-04-13       |170                       |Major (1-4 hrs)     |true                   |
|SA-SITE-0742|VEND_C|2024-04-25       |64                        |Major (1-4 hrs)     |true                   |
|SA-SITE-0217|VEND_C|2024-04-02       |180                       |Major (1-4 hrs)     |true                   |
|SA-SITE-0615|VEND_C|2024-04-08       |133                       |Major (1-4 hrs)     |true                   |
|SA-SITE-0610|VEND_C|2024-04-12

In [2]:
df = spark.sql("SELECT * FROM lh_Silver_Telecom.dbo.silver_network_events LIMIT 10")
display(df)

StatementMeta(, 45cca8f8-1576-4691-a588-aba45c42f207, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 05fa17b2-51e0-4362-92f8-2675f64b4452)